In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
df = pd.read_csv("/content/drive/MyDrive/ICBHI17/Meta data/file_list.csv")

In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
df = pd.read_csv("/content/drive/MyDrive/ICBHI17/Meta data/file_list.csv")

# 2. Initialize StratifiedGroupKFold for 3 folds
sgkf = StratifiedGroupKFold(n_splits=5)

# 3. Create a new column 'Fold' and assign fold numbers
df['Fold'] = -1  # Default value

# The split function takes (data, labels to stratify by, groups to keep separate)
# Here, we stratify by 'ID' and group by 'Subject'
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['ID'], groups=df['Subject'])):
    df.loc[val_idx, 'Fold'] = fold + 1  # 1, 2, or 3

# Optional: Print the summary to verify the balancing
print("Total files per fold:")
print(df['Fold'].value_counts().sort_index())
print("\nDistribution of IDs per fold:")
print(df.groupby(['Fold', 'ID']).size())

# 4. Save the new metadata to a CSV file
df.to_csv("file_list_with_folds_5.csv", index=False)
print("\nFile successfully saved as 'file_list_with_folds_5.csv'")

Total files per fold:
Fold
1    587
2    590
3    598
4    607
5    596
Name: count, dtype: int64

Distribution of IDs per fold:
Fold  ID    
1     bri         6
      brx         9
      copd      521
      hlth       21
      pna        15
      urti       15
2     asthma      3
      bri         9
      brx         9
      copd      518
      hlth       21
      lrti        3
      pna        12
      urti       15
3     bri         9
      brx         6
      copd      520
      hlth       21
      pna        27
      urti       15
4     bri         6
      brx         6
      copd      520
      hlth       21
      lrti        3
      pna        39
      urti       12
5     bri         9
      brx        18
      copd      518
      hlth       21
      pna        18
      urti       12
dtype: int64

File successfully saved as 'file_list_with_folds_5.csv'


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


In [7]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
# df = pd.read_csv("file_list_final.csv")

# Exclude 'lrt' and 'ast' IDs from the dataset and reset the index
df = df[~df['ID'].isin(['lrt', 'ast'])].reset_index(drop=True)

# 2. Initialize StratifiedGroupKFold for 5 folds
sgkf = StratifiedGroupKFold(n_splits=5)

# 3. Create a new column 'Fold' and assign fold numbers
df['Fold'] = -1  # Default value

# The split function takes (data, labels to stratify by, groups to keep separate)
# Here, we stratify by 'ID' and group by 'Subject'
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['ID'], groups=df['Subject'])):
    df.loc[val_idx, 'Fold'] = fold + 1  # 1, 2, 3, 4, or 5

# Optional: Print the summary to verify the balancing
print("Total files per fold:")
print(df['Fold'].value_counts().sort_index())
print("\nDistribution of IDs per fold:")
print(df.groupby(['Fold', 'ID']).size())

# 4. Save the new metadata to a CSV file
output_filename = "file_list_with_folds_5_lrt_ast.csv"
df.to_csv(output_filename, index=False)
print(f"\nFile successfully saved as '{output_filename}'")

Total files per fold:
Fold
1    587
2    590
3    598
4    607
5    596
Name: count, dtype: int64

Distribution of IDs per fold:
Fold  ID    
1     bri         6
      brx         9
      copd      521
      hlth       21
      pna        15
      urti       15
2     asthma      3
      bri         9
      brx         9
      copd      518
      hlth       21
      lrti        3
      pna        12
      urti       15
3     bri         9
      brx         6
      copd      520
      hlth       21
      pna        27
      urti       15
4     bri         6
      brx         6
      copd      520
      hlth       21
      lrti        3
      pna        39
      urti       12
5     bri         9
      brx        18
      copd      518
      hlth       21
      pna        18
      urti       12
dtype: int64

File successfully saved as 'file_list_with_folds_5_lrt_ast.csv'


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


In [8]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
# df = pd.read_csv("file_list_final.csv")

# Exclude 'lrt' and 'ast' IDs from the dataset and reset the index
df = df[~df['ID'].isin(['lrti', 'asthma'])].reset_index(drop=True)

# 2. Initialize StratifiedGroupKFold for 5 folds
sgkf = StratifiedGroupKFold(n_splits=5)

# 3. Create a new column 'Fold' and assign fold numbers
df['Fold'] = -1  # Default value

# The split function takes (data, labels to stratify by, groups to keep separate)
# Here, we stratify by 'ID' and group by 'Subject'
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['ID'], groups=df['Subject'])):
    df.loc[val_idx, 'Fold'] = fold + 1  # 1, 2, 3, 4, or 5

# --- NEW STEP: Reduce 'cop' files so each fold has exactly the same amount ---

# Find the minimum number of 'cop' files across all folds (This ensures we keep as much data as possible)
cop_counts = df[df['ID'] == 'copd'].groupby(df['Fold']).size()
min_cop = cop_counts.min()

print(f"Original 'cop' counts per fold:\n{cop_counts.to_string()}")
print(f"\nTarget 'cop' count per fold will be set to: {min_cop}\n")

# Create an empty list to store the rows we want to keep
rows_to_keep = []

for fold in range(1, 6):
    # Get all 'cop' files for this specific fold, and randomly sample exactly 'min_cop' amount
    cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] == 'copd')]
    cop_sampled = cop_in_fold.sample(n=min_cop, random_state=42) # random_state ensures reproducibility
    rows_to_keep.append(cop_sampled)

    # Get all non-'cop' files for this fold and keep ALL of them
    non_cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] != 'copd')]
    rows_to_keep.append(non_cop_in_fold)

# Combine everything back into a single balanced DataFrame
df = pd.concat(rows_to_keep).reset_index(drop=True)

# -----------------------------------------------------------------------------

# Optional: Print the summary to verify the balancing
print("Total files per fold:")
print(df['Fold'].value_counts().sort_index())
print("\nDistribution of IDs per fold:")
print(df.groupby(['Fold', 'ID']).size())

# 4. Save the new metadata to a CSV file
output_filename = "file_list_with_folds_5_lrt_ast._equal_cop.csv"
df.to_csv(output_filename, index=False)
print(f"\nFile successfully saved as '{output_filename}'")

Original 'cop' counts per fold:
Fold
1    521
2    518
3    520
4    520
5    518

Target 'cop' count per fold will be set to: 518

Total files per fold:
Fold
1    584
2    584
3    596
4    602
5    596
Name: count, dtype: int64

Distribution of IDs per fold:
Fold  ID  
1     bri       6
      brx       9
      copd    518
      hlth     21
      pna      15
      urti     15
2     bri       9
      brx       9
      copd    518
      hlth     21
      pna      12
      urti     15
3     bri       9
      brx       6
      copd    518
      hlth     21
      pna      27
      urti     15
4     bri       6
      brx       6
      copd    518
      hlth     21
      pna      39
      urti     12
5     bri       9
      brx      18
      copd    518
      hlth     21
      pna      18
      urti     12
dtype: int64

File successfully saved as 'file_list_with_folds_5_lrt_ast._equal_cop.csv'


In [9]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
# df = pd.read_csv("file_list_final.csv")

# Exclude 'lrt' and 'ast' IDs from the dataset and reset the index
df = df[~df['ID'].isin(['lrti', 'asthma'])].reset_index(drop=True)

# 2. Initialize StratifiedGroupKFold for 5 folds
sgkf = StratifiedGroupKFold(n_splits=5)

# 3. Create a new column 'Fold' and assign fold numbers
df['Fold'] = -1

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['ID'], groups=df['Subject'])):
    df.loc[val_idx, 'Fold'] = fold + 1

# --- NEW STEP: Drastically reduce 'cop' to match the other classes ---
rows_to_keep = []

for fold in range(1, 6):
    # 1. Get all the minority classes (non-'cop' files) for this fold
    non_cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] != 'copd')]
    rows_to_keep.append(non_cop_in_fold)

    # 2. Find the highest count among the other classes to set as our target
    # (e.g., if 'pne' has 39 files in this fold, we will reduce 'cop' to 39 files)
    target_cop_count = non_cop_in_fold['ID'].value_counts().max()

    # 3. Get the 'cop' files for this fold
    cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] == 'copd')]

    # 4. Randomly sample the 'cop' files to match the target count (skipping the rest)
    if len(cop_in_fold) > target_cop_count:
        cop_sampled = cop_in_fold.sample(n=target_cop_count, random_state=42)
    else:
        cop_sampled = cop_in_fold

    rows_to_keep.append(cop_sampled)

# Combine everything back into a single dataframe
df_balanced = pd.concat(rows_to_keep).reset_index(drop=True)

# -----------------------------------------------------------------------------

# Print the summary to verify the new balanced distribution
print("Total files per fold:")
print(df_balanced['Fold'].value_counts().sort_index())

print("\nDistribution of IDs per fold:")
print(df_balanced.groupby(['Fold', 'ID']).size())

# Save the new metadata to a CSV file
output_filename = "file_list_with_folds_5_balanced.csv"
df_balanced.to_csv(output_filename, index=False)
print(f"\nFile successfully saved as '{output_filename}'")

Total files per fold:
Fold
1     99
2    102
3    123
4     90
5     87
Name: count, dtype: int64

Distribution of IDs per fold:
Fold  ID  
1     bri      9
      brx     18
      copd    21
      hlth    21
      pna     18
      urti    12
2     bri      6
      brx      6
      copd    27
      hlth    21
      pna     27
      urti    15
3     bri      6
      brx      6
      copd    39
      hlth    21
      pna     39
      urti    12
4     bri      9
      brx      9
      copd    21
      hlth    21
      pna     15
      urti    15
5     bri      9
      brx      9
      copd    21
      hlth    21
      pna     12
      urti    15
dtype: int64

File successfully saved as 'file_list_with_folds_5_balanced.csv'


In [10]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load the dataset
# df = pd.read_csv("file_list_final.csv")

# Exclude 'lrt' and 'ast' IDs from the dataset and reset the index
df = df[~df['ID'].isin(['lrti', 'asthma'])].reset_index(drop=True)

# 2. Initialize StratifiedGroupKFold for 5 folds
sgkf = StratifiedGroupKFold(n_splits=5)

# 3. Create a new column 'Fold' and assign fold numbers
df['Fold'] = -1

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['ID'], groups=df['Subject'])):
    df.loc[val_idx, 'Fold'] = fold + 1

# --- NEW STEP: Drastically reduce 'cop' to match the other classes ---
rows_to_keep = []

for fold in range(1, 6):
    # 1. Get all the minority classes (non-'cop' files) for this fold
    non_cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] != 'copd')]
    rows_to_keep.append(non_cop_in_fold)

    # 2. Find the highest count among the other classes to set as our target
    # (e.g., if 'pne' has 39 files in this fold, we will reduce 'cop' to 39 files)
    target_cop_count = non_cop_in_fold['ID'].value_counts().max()

    # 3. Get the 'cop' files for this fold
    cop_in_fold = df[(df['Fold'] == fold) & (df['ID'] == 'copd')]

    # 4. Randomly sample the 'cop' files to match the target count (skipping the rest)
    if len(cop_in_fold) > target_cop_count:
        cop_sampled = cop_in_fold.sample(n=target_cop_count, random_state=420)
    else:
        cop_sampled = cop_in_fold

    rows_to_keep.append(cop_sampled)

# Combine everything back into a single dataframe
df_balanced = pd.concat(rows_to_keep).reset_index(drop=True)

# -----------------------------------------------------------------------------

# Print the summary to verify the new balanced distribution
print("Total files per fold:")
print(df_balanced['Fold'].value_counts().sort_index())

print("\nDistribution of IDs per fold:")
print(df_balanced.groupby(['Fold', 'ID']).size())

# Save the new metadata to a CSV file
output_filename = "file_list_with_folds_5_balanced.csv"
df_balanced.to_csv(output_filename, index=False)
print(f"\nFile successfully saved as '{output_filename}'")

Total files per fold:
Fold
1     99
2    102
3    123
4     90
5     87
Name: count, dtype: int64

Distribution of IDs per fold:
Fold  ID  
1     bri      9
      brx     18
      copd    21
      hlth    21
      pna     18
      urti    12
2     bri      6
      brx      6
      copd    27
      hlth    21
      pna     27
      urti    15
3     bri      6
      brx      6
      copd    39
      hlth    21
      pna     39
      urti    12
4     bri      9
      brx      9
      copd    21
      hlth    21
      pna     15
      urti    15
5     bri      9
      brx      9
      copd    21
      hlth    21
      pna     12
      urti    15
dtype: int64

File successfully saved as 'file_list_with_folds_5_balanced.csv'


In [11]:
import glob
import os

# Define the path to your folder (use "." for the current folder)
folder_path = "/content/drive/MyDrive/ICBHI17/segmented"

# Search for all .wav files
search_pattern = os.path.join(folder_path, "*.wav")
wav_files = glob.glob(search_pattern)

print(f"Found {len(wav_files)} .wav files.")

# Print the first 5 files found
for file in wav_files[:5]:
    print(file)

Found 2978 .wav files.
/content/drive/MyDrive/ICBHI17/segmented/181_1b1_Ar_mc_LittC2SE_2_copd.wav
/content/drive/MyDrive/ICBHI17/segmented/181_1b1_Tc_mc_LittC2SE_1_copd.wav
/content/drive/MyDrive/ICBHI17/segmented/180_1b4_Al_mc_AKGC417L_2_copd.wav
/content/drive/MyDrive/ICBHI17/segmented/178_1b6_Lr_mc_AKGC417L_3_copd.wav
/content/drive/MyDrive/ICBHI17/segmented/177_2b4_Pl_mc_AKGC417L_1_copd.wav


In [12]:
import os

# Define the folder and the filename
folder_path = "/content/drive/MyDrive/ICBHI17/segmented"
filename = "120_1b1_Pl_sc_Meditron_4_copd.wav"

# Combine them into a full path
full_path = os.path.join(folder_path, filename)

# Check if the file exists
if os.path.exists(full_path):
    print(full_path)
    print(f"Success: {filename} exists!")
else:
    print(f"Error: {filename} was not found in the folder.")

/content/drive/MyDrive/ICBHI17/segmented/120_1b1_Pl_sc_Meditron_4_copd.wav
Success: 120_1b1_Pl_sc_Meditron_4_copd.wav exists!


In [13]:
import pandas as pd
import librosa
import numpy as np
import os

# 1. Load the metadata
csv_filename = "/content/drive/MyDrive/ICBHI17/Meta data/file_list_with_folds_5_balanced.csv"  # Update if using the balanced fold CSV
df = pd.read_csv(csv_filename)

# Clean up column names in case there are trailing spaces (like 'Filename ')
df.columns = df.columns.str.strip()

# Set the folder where your .wav files are located
# (Leave as "" if they are in the same folder as the script)
audio_dir = "/content/drive/MyDrive/ICBHI17/segmented"

# 2. Initialize the feature matrix X and label vector u
X = []
u = []

print(f"Starting feature extraction for {len(df)} files...")

# 3. Iterate through each row in the CSV
for index, row in df.iterrows():
    # Keep track of progress
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

    filename = row['Filename']
    label = row['ID']

    # Construct the full path to the audio file
    file_path = os.path.join(audio_dir, filename)
    file_path = file_path.strip()
    # u.append(label)
    print(file_path)
    # Check if file exists to avoid crashing
    # if not os.path.exists(file_path):
    #     print(f"Warning: {file_path} not found. Skipping.")
    #     continue
    if os.path.exists(file_path):
        print(f"Success: {filename} exists!")

    try:
        # Load audio file (sr=22050 is standard, use sr=None for original sample rate)
        y, sr = librosa.load(file_path, sr=22050)

        # --- Feature Extraction ---

        # A. Log Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_features = np.mean(log_mel_spec.T, axis=0)  # Average across time

        # B. Constant-Q Transform (CQT)
        cqt_spec = np.abs(librosa.cqt(y=y, sr=sr))
        log_cqt_spec = librosa.amplitude_to_db(cqt_spec, ref=np.max)
        cqt_features = np.mean(log_cqt_spec.T, axis=0)  # Average across time

        # C. Chroma Features
        chroma_spec = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_features = np.mean(chroma_spec.T, axis=0)  # Average across time

        # Combine all features into a single 1D array for this file
        combined_features = np.hstack((mel_features, cqt_features, chroma_features))
        print(combined_features)
        # 4. Append to X and u
        X.append(combined_features)
        u.append(label)


    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 5. Convert lists to numpy arrays for machine learning
X = np.array(X)
u = np.array(u)

print("\n--- Extraction Complete ---")
print(f"Total samples processed: {len(X)}")
print(f"Shape of feature matrix X: {X.shape}")
print(f"Shape of label array u: {u.shape}")

Starting feature extraction for 501 files...
/content/drive/MyDrive/ICBHI17/segmented/111_1b2_Tc_sc_Meditron_1_brx.wav
Success: 111_1b2_Tc_sc_Meditron_1_brx.wav  exists!
[-17.371202   -19.875834   -24.203825   -27.963497   -30.121958
 -32.083504   -33.019005   -34.3836     -35.583714   -36.465714
 -37.3662     -37.717903   -38.318375   -39.636204   -39.54739
 -39.705997   -39.349354   -38.785843   -38.731888   -38.460392
 -39.462074   -40.99503    -42.293777   -42.853462   -42.960213
 -44.541916   -43.871853   -42.69253    -41.21085    -39.7471
 -39.836514   -40.799606   -41.434135   -42.756424   -42.78472
 -42.94988    -42.624672   -42.20701    -42.913815   -41.609814
 -40.595432   -38.802914   -39.904644   -39.806004   -40.61562
 -41.92431    -42.585667   -42.329964   -42.39239    -41.676266
 -40.917168   -41.41424    -42.622528   -43.495846   -43.837524
 -43.766567   -43.859257   -45.566376   -47.83865    -50.864464
 -52.219166   -53.76192    -54.508633   -54.959057   -55.82936
 -57

/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Streaming output truncated to the last 5000 lines.
 -48.8832     -47.939384   -48.428387   -50.315495   -52.126953
 -54.32477    -56.122684   -55.419872   -53.535942   -52.463284
 -53.59436    -56.742485   -58.666534   -59.589264   -60.69068
 -60.353226   -58.642544   -57.97014    -60.114365   -63.617302
 -64.188446   -63.039146   -64.47493    -67.34534    -70.12686
 -71.886314   -72.21298    -73.47719    -73.648346   -73.51599
 -73.78866    -74.61947    -74.73965    -75.65303    -75.41864
 -75.40844    -75.1064       0.7518065    0.7409302    0.7388418
   0.768731     0.7908697    0.8044864    0.76811725   0.7718011
   0.7853439    0.78697896   0.74266595   0.7205571 ]
/content/drive/MyDrive/ICBHI17/segmented/176_1b3_Pl_mc_AKGC417L_1_copd.wav
Success: 176_1b3_Pl_mc_AKGC417L_1_copd.wav  exists!
[-10.6949415  -17.127415   -24.027912   -29.99118    -32.990925
 -34.788063   -37.078842   -40.103474   -41.705624   -42.67013
 -44.772835   -45.89469    -46.03496    -47.118313   -47.08528
 -49

In [14]:
X

array([[-17.371202  , -19.875834  , -24.203825  , ...,   0.6759285 ,
          0.657655  ,   0.669894  ],
       [-16.225758  , -19.452269  , -24.031197  , ...,   0.6176883 ,
          0.6267607 ,   0.663498  ],
       [-15.867942  , -20.79792   , -25.765026  , ...,   0.66451585,
          0.6429498 ,   0.63964236],
       ...,
       [-10.436033  , -17.666283  , -24.750488  , ...,   0.73807555,
          0.7077591 ,   0.70945436],
       [-16.115263  , -26.941046  , -33.38586   , ...,   0.7189293 ,
          0.6858374 ,   0.6827208 ],
       [ -7.9610033 , -14.183689  , -18.654541  , ...,   0.7024147 ,
          0.6891343 ,   0.7060116 ]], dtype=float32)

In [15]:
u

array(['brx', 'brx', 'brx', 'brx', 'brx', 'brx', 'hlth', 'hlth', 'hlth',
       'hlth', 'hlth', 'hlth', 'hlth', 'hlth', 'hlth', 'bri', 'bri',
       'bri', 'bri', 'bri', 'bri', 'urti', 'urti', 'urti', 'hlth', 'hlth',
       'hlth', 'urti', 'urti', 'urti', 'urti', 'urti', 'urti', 'urti',
       'urti', 'urti', 'urti', 'urti', 'urti', 'hlth', 'hlth', 'hlth',
       'hlth', 'hlth', 'hlth', 'brx', 'brx', 'brx', 'hlth', 'hlth',
       'hlth', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna',
       'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'copd', 'copd',
       'copd', 'copd', 'copd', 'copd', 'copd', 'copd', 'copd', 'copd',
       'copd', 'copd', 'copd', 'copd', 'copd', 'copd', 'copd', 'copd',
       'copd', 'copd', 'copd', 'brx', 'brx', 'brx', 'brx', 'brx', 'brx',
       'hlth', 'hlth', 'hlth', 'urti', 'urti', 'urti', 'pna', 'pna',
       'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna', 'pna',
       'pna', 'urti', 'urti', 'urti', 'hlth', 'hlth', 'hlth', 'hlth',
      

In [16]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# CLASS MAP - FOFR INTEGER BASED REP - TO AVOID CASE SENSIVITITY - COPD - 1 , BROX -2 --- EASY MAPPING
# 1. Encode text labels in 'u' to integers (0, 1, 2, 3...)
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)

# Note: to get the original names back later, you can use encoder.classes_
print("Classes mapping:", dict(enumerate(encoder.classes_)))

# 2. Split into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, u_encoded, test_size=0.2, random_state=42)

# 3. Scale the features (Important for SVMs, Neural Networks, etc.)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Classes mapping: {0: np.str_('bri'), 1: np.str_('brx'), 2: np.str_('copd'), 3: np.str_('hlth'), 4: np.str_('pna'), 5: np.str_('urti')}


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

print("--- Training Classical Models ---")

# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_predictions = rf_model.predict(X_test_scaled)

print("\nRandom Forest Accuracy:", accuracy_score(y_test, rf_predictions))

# --- Support Vector Machine (SVM) ---
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_predictions = svm_model.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, svm_predictions))
print("\nSVM Classification Report:\n", classification_report(y_test, svm_predictions, target_names=encoder.classes_))

--- Training Classical Models ---

Random Forest Accuracy: 0.693069306930693
SVM Accuracy: 0.7227722772277227

SVM Classification Report:
               precision    recall  f1-score   support

         bri       1.00      0.25      0.40         4
         brx       1.00      0.82      0.90        11
        copd       0.94      0.83      0.88        35
        hlth       0.52      0.61      0.56        18
         pna       0.64      1.00      0.78        21
        urti       0.33      0.17      0.22        12

    accuracy                           0.72       101
   macro avg       0.74      0.61      0.62       101
weighted avg       0.74      0.72      0.71       101



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
#iteration
print("--- Training 1D CNN ---")

# 1. Reshape data from 2D to 3D for the Conv1D layer
# Current shape: (samples, features) -> New shape: (samples, features, 1)
X_train_cnn = np.expand_dims(X_train_scaled, axis=2)
X_test_cnn = np.expand_dims(X_test_scaled, axis=2)

print(f"CNN Input Shape: {X_train_cnn.shape}")

# 2. Determine number of classes for the output layer
num_classes = len(encoder.classes_)

# 3. Build the 1D CNN Architecture
model = Sequential([
    # First Convolutional Block
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),

    # Second Convolutional Block
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    # Flatten and pass to Dense (Fully Connected) layers
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5), # Helps prevent overfitting

    # Output layer (softmax for multi-class classification)
    Dense(num_classes, activation='softmax')
])

# 4. Compile the model
# We use 'sparse_categorical_crossentropy' because our labels are integers (not one-hot encoded)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Train the model
history = model.fit(X_train_cnn, y_train,
                    epochs=30,
                    batch_size=32,
                    validation_data=(X_test_cnn, y_test),
                    verbose=1)

# 6. Evaluate the model
test_loss, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\n1D CNN Test Accuracy: {test_acc:.4f}")

--- Training 1D CNN ---
CNN Input Shape: (400, 224, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.2915 - loss: 1.6958 - val_accuracy: 0.5050 - val_loss: 1.4700
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.5192 - loss: 1.3613 - val_accuracy: 0.5347 - val_loss: 1.2036
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.5020 - loss: 1.2015 - val_accuracy: 0.6139 - val_loss: 1.0460
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.6709 - loss: 0.9510 - val_accuracy: 0.6436 - val_loss: 0.9615
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.6571 - loss: 0.8852 - val_accuracy: 0.6634 - val_loss: 0.9228
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.7098 - loss: 0.8122 - val_accuracy: 0.7129 - val_loss: 0.7522
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.7162 - loss: 0.7605 - val_accuracy: 0.7426 - val_loss: 0.7257
Epoch 8/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.7619 - loss: 0.6983 - val_accuracy: 0.7525 - v

FEATURE CHECKING  fold

In [ ]:
import pandas as pd
import librosa
import numpy as np
import os

# 1. Load the metadata
csv_filename = "/content/drive/MyDrive/ICBHI17/Meta data/file_list_with_folds_5_balanced.csv"
df = pd.read_csv(csv_filename)
df.columns = df.columns.str.strip()

audio_dir = "/content/drive/MyDrive/ICBHI17/segmented"

# 2. Initialize the feature matrix X, label vector u, and folds list
X = []
u = []
folds_list = []  # <-- NEW: Track the fold of each file

print(f"Starting feature extraction for {len(df)} files...")

# 3. Iterate through each row in the CSV
for index, row in df.iterrows():
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

    filename = row['Filename']
    label = row['ID']
    fold_num = row['Fold'] # <-- NEW: Get the fold number

    file_path = os.path.join(audio_dir, filename).strip()

    if not os.path.exists(file_path):
        # We should skip if not found so we don't crash
        print(f"Warning: {file_path} not found. Skipping.")
        continue

    try:
        y, sr = librosa.load(file_path, sr=22050)

        # A. Log Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_features = np.mean(log_mel_spec.T, axis=0)

        # B. Constant-Q Transform (CQT)
        cqt_spec = np.abs(librosa.cqt(y=y, sr=sr))
        log_cqt_spec = librosa.amplitude_to_db(cqt_spec, ref=np.max)
        cqt_features = np.mean(log_cqt_spec.T, axis=0)

        # C. Chroma Features
        chroma_spec = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_features = np.mean(chroma_spec.T, axis=0)

        combined_features = np.hstack((mel_features, cqt_features, chroma_features))

        # 4. Append to lists ONLY upon success
        X.append(combined_features)
        u.append(label)
        folds_list.append(fold_num) # <-- NEW: Append the fold

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 5. Convert lists to numpy arrays
X = np.array(X)
u = np.array(u)
folds_array = np.array(folds_list) # <-- NEW: Convert folds to array

print("\n--- Extraction Complete ---")
print(f"Shape of X: {X.shape}")
print(f"Shape of u: {u.shape}")
print(f"Shape of folds: {folds_array.shape}")





Starting feature extraction for 501 files...


/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed 50 / 501 files...
Processed 100 / 501 files...
Processed 150 / 501 files...
Processed 200 / 501 files...
Processed 250 / 501 files...
Processed 300 / 501 files...
Processed 350 / 501 files...
Processed 400 / 501 files...
Processed 450 / 501 files...
Processed 500 / 501 files...

--- Extraction Complete ---
Shape of X: (501, 224)
Shape of u: (501,)
Shape of folds: (501,)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
# 1. Encode text labels to integers
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)

# Array to store the accuracy of each fold
fold_accuracies = []

print("\n--- Starting 5-Fold Cross Validation ---")

# 2. Loop through the 5 folds
for test_fold in range(1, 6):
    print(f"\n--- Processing Fold {test_fold} as Test Set ---")

    # Create boolean masks to separate train and test data
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    # Split the data into Train (4 folds) and Test (1 fold)
    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples.")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and Train the model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Test the model
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    # Test the model
    predictions = model.predict(X_test_scaled)

    # Accuracy
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)
    print("Confusion Matrix:")
    print(cm)

    # Classification Report
    print("Classification Report:")
    print(classification_report(y_test, predictions))
    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

# Final Results
print("\n--- Subject-aware Final Results ---")
print(f"Accuracies across the 5 folds: {[round(a, 4) for a in fold_accuracies]}")
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f}")


# Random based


rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 5-fold cross validation
scores = cross_val_score(rf, X,u, cv=5)
print("\n--- Random Split Final Results ---")
print("Accuracy for each fold:", scores)
print("Average Accuracy:", scores.mean())
# Train model
rf.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf.predict(X_test_scaled)

# Accuracy

# X_train, X_test, y_train, y_test = train_test_split(X, u_encoded, test_size=0.2, random_state=42)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# print("\n--- Random Split Final Results ---")
# print("Accuracy:", accuracy_score(y_test, y_pred))


# # Confusion Matrix
# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

# # Classification Report
# print("Classification Report:")
# print(classification_report(y_test, y_pred))


--- Starting 5-Fold Cross Validation ---

--- Processing Fold 1 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold 1: 0.5402
Confusion Matrix:
[[ 0  0  0  2  3  1]
 [ 0  0  9  0  0  0]
 [ 0  1 18  1  1  0]
 [ 0  0  1 15  4  1]
 [ 0  0  0  1 14  0]
 [ 4  1  2  2  6  0]]
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.00      0.00      0.00         9
           2       0.60      0.86      0.71        21
           3       0.71      0.71      0.71        21
           4       0.50      0.93      0.65        15
           5       0.00      0.00      0.00        15

    accuracy                           0.54        87
   macro avg       0.30      0.42      0.35        87
weighted avg       0.40      0.54      0.46        87

Accuracy for Fold 1: 0.5402

--- Processing Fold 2 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold

Without chroma - check accuracy on fold

In [ ]:
import pandas as pd
import librosa
import numpy as np
import os

# 1. Load the metadata
csv_filename = "/content/drive/MyDrive/ICBHI17/Meta data/file_list_with_folds_5_balanced.csv"
df = pd.read_csv(csv_filename)
df.columns = df.columns.str.strip()

audio_dir = "/content/drive/MyDrive/ICBHI17/segmented"

# 2. Initialize the feature matrix X, label vector u, and folds list
X = []
u = []
folds_list = []  # <-- NEW: Track the fold of each file

print(f"Starting feature extraction for {len(df)} files...")

# 3. Iterate through each row in the CSV
for index, row in df.iterrows():
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

    filename = row['Filename']
    label = row['ID']
    fold_num = row['Fold'] # <-- NEW: Get the fold number

    file_path = os.path.join(audio_dir, filename).strip()

    if not os.path.exists(file_path):
        # We should skip if not found so we don't crash
        print(f"Warning: {file_path} not found. Skipping.")
        continue

    try:
      #y wave form
        y, sr = librosa.load(file_path, sr=22050) #sample rate

        # A. Log Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_features = np.mean(log_mel_spec.T, axis=0)

        # B. Constant-Q Transform (CQT)
        cqt_spec = np.abs(librosa.cqt(y=y, sr=sr))
        log_cqt_spec = librosa.amplitude_to_db(cqt_spec, ref=np.max)
        cqt_features = np.mean(log_cqt_spec.T, axis=0)

        # C. Chroma Features
        # chroma_spec = librosa.feature.chroma_stft(y=y, sr=sr)
        # chroma_features = np.mean(chroma_spec.T, axis=0)

        # combined_features = np.hstack((mel_features, cqt_features, chroma_features))
        combined_features = np.hstack((mel_features, cqt_features))


        # 4. Append to lists ONLY upon success
        X.append(combined_features)
        u.append(label)
        folds_list.append(fold_num) # <-- NEW: Append the fold

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 5. Convert lists to numpy arrays
X = np.array(X)
u = np.array(u)
folds_array = np.array(folds_list) # <-- NEW: Convert folds to array

print("\n--- Extraction Complete ---")
print(f"Shape of X: {X.shape}")
print(f"Shape of u: {u.shape}")
print(f"Shape of folds: {folds_array.shape}")

Starting feature extraction for 501 files...
Processed 50 / 501 files...
Processed 100 / 501 files...
Processed 150 / 501 files...
Processed 200 / 501 files...
Processed 250 / 501 files...
Processed 300 / 501 files...
Processed 350 / 501 files...
Processed 400 / 501 files...
Processed 450 / 501 files...
Processed 500 / 501 files...

--- Extraction Complete ---
Shape of X: (501, 212)
Shape of u: (501,)
Shape of folds: (501,)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Encode text labels to integers
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)

# Array to store the accuracy of each fold
fold_accuracies = []

print("\n--- Starting 5-Fold Cross Validation ---")

# 2. Loop through the 5 folds
for test_fold in range(1, 6):
    print(f"\n--- Processing Fold {test_fold} as Test Set ---")

    # Create boolean masks to separate train and test data
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    # Split the data into Train (4 folds) and Test (1 fold)
    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples.")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and Train the model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Test the model
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

# Final Results
print("\n--- Final Results ---")
print(f"Accuracies across the 5 folds: {[round(a, 4) for a in fold_accuracies]}")
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f}")


--- Starting 5-Fold Cross Validation ---

--- Processing Fold 1 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold 1: 0.5632

--- Processing Fold 2 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold 2: 0.4138

--- Processing Fold 3 as Test Set ---
Training on 396 samples, Testing on 105 samples.
Accuracy for Fold 3: 0.6000

--- Processing Fold 4 as Test Set ---
Training on 378 samples, Testing on 123 samples.
Accuracy for Fold 4: 0.5122

--- Processing Fold 5 as Test Set ---
Training on 402 samples, Testing on 99 samples.
Accuracy for Fold 5: 0.4040

--- Final Results ---
Accuracies across the 5 folds: [0.5632, 0.4138, 0.6, 0.5122, 0.404]
Average Accuracy: 0.4986


MEL
**bold text**

In [ ]:
import pandas as pd
import librosa
import numpy as np
import os

# 1. Load the metadata
csv_filename = "/content/drive/MyDrive/ICBHI17/Meta data/file_list_with_folds_5_balanced.csv"
df = pd.read_csv(csv_filename)
df.columns = df.columns.str.strip()

audio_dir = "/content/drive/MyDrive/ICBHI17/segmented"

# 2. Initialize the feature matrix X, label vector u, and folds list
X = []
u = []
folds_list = []  # <-- NEW: Track the fold of each file

print(f"Starting feature extraction for {len(df)} files...")

# 3. Iterate through each row in the CSV
for index, row in df.iterrows():
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

    filename = row['Filename']
    label = row['ID']
    fold_num = row['Fold'] # <-- NEW: Get the fold number

    file_path = os.path.join(audio_dir, filename).strip()

    if not os.path.exists(file_path):
        # We should skip if not found so we don't crash
        print(f"Warning: {file_path} not found. Skipping.")
        continue

    try:
      #y wave form
        y, sr = librosa.load(file_path, sr=22050) #sample rate

        # A. Log Mel Spectrogram
        # mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        # log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        # mel_features = np.mean(log_mel_spec.T, axis=0)

        # B. Constant-Q Transform (CQT)
        cqt_spec = np.abs(librosa.cqt(y=y, sr=sr))
        log_cqt_spec = librosa.amplitude_to_db(cqt_spec, ref=np.max)
        cqt_features = np.mean(log_cqt_spec.T, axis=0)

        # C. Chroma Features
        chroma_spec = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_features = np.mean(chroma_spec.T, axis=0)

        # combined_features = np.hstack((mel_features, cqt_features, chroma_features))
        combined_features = np.hstack((cqt_features,chroma_features))


        # 4. Append to lists ONLY upon success
        X.append(combined_features)
        u.append(label)
        folds_list.append(fold_num) # <-- NEW: Append the fold

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 5. Convert lists to numpy arrays
X = np.array(X)
u = np.array(u)
folds_array = np.array(folds_list) # <-- NEW: Convert folds to array

print("\n--- Extraction Complete ---")
print(f"Shape of X: {X.shape}")
print(f"Shape of u: {u.shape}")
print(f"Shape of folds: {folds_array.shape}")

Starting feature extraction for 501 files...


/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed 50 / 501 files...
Processed 100 / 501 files...
Processed 150 / 501 files...
Processed 200 / 501 files...
Processed 250 / 501 files...
Processed 300 / 501 files...
Processed 350 / 501 files...
Processed 400 / 501 files...
Processed 450 / 501 files...
Processed 500 / 501 files...

--- Extraction Complete ---
Shape of X: (501, 96)
Shape of u: (501,)
Shape of folds: (501,)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Encode text labels to integers
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)

# Array to store the accuracy of each fold
fold_accuracies = []

print("\n--- Starting 5-Fold Cross Validation ---")

# 2. Loop through the 5 folds
for test_fold in range(1, 6):
    print(f"\n--- Processing Fold {test_fold} as Test Set ---")

    # Create boolean masks to separate train and test data
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    # Split the data into Train (4 folds) and Test (1 fold)
    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples.")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and Train the model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Test the model
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

# Final Results
print("\n--- Final Results ---")
print(f"Accuracies across the 5 folds: {[round(a, 4) for a in fold_accuracies]}")
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f}")


--- Starting 5-Fold Cross Validation ---

--- Processing Fold 1 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold 1: 0.5172

--- Processing Fold 2 as Test Set ---
Training on 414 samples, Testing on 87 samples.
Accuracy for Fold 2: 0.4368

--- Processing Fold 3 as Test Set ---
Training on 396 samples, Testing on 105 samples.
Accuracy for Fold 3: 0.4571

--- Processing Fold 4 as Test Set ---
Training on 378 samples, Testing on 123 samples.
Accuracy for Fold 4: 0.5854

--- Processing Fold 5 as Test Set ---
Training on 402 samples, Testing on 99 samples.
Accuracy for Fold 5: 0.4040

--- Final Results ---
Accuracies across the 5 folds: [0.5172, 0.4368, 0.4571, 0.5854, 0.404]
Average Accuracy: 0.4801


CQT

In [ ]:
import pandas as pd
import librosa
import numpy as np
import os

# 1. Load the metadata
csv_filename = "/content/drive/MyDrive/ICBHI17/Meta data/file_list_with_folds_5_balanced.csv"
df = pd.read_csv(csv_filename)
df.columns = df.columns.str.strip()

audio_dir = "/content/drive/MyDrive/ICBHI17/segmented"

# 2. Initialize the feature matrix X, label vector u, and folds list
X = []
u = []
folds_list = []  # <-- NEW: Track the fold of each file

print(f"Starting feature extraction for {len(df)} files...")

# 3. Iterate through each row in the CSV
for index, row in df.iterrows():
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

    filename = row['Filename']
    label = row['ID']
    fold_num = row['Fold'] # <-- NEW: Get the fold number

    file_path = os.path.join(audio_dir, filename).strip()

    if not os.path.exists(file_path):
        # We should skip if not found so we don't crash
        print(f"Warning: {file_path} not found. Skipping.")
        continue

    try:
      #y wave form
        y, sr = librosa.load(file_path, sr=22050) #sample rate

        # A. Log Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        mel_features = np.mean(log_mel_spec.T, axis=0)

        # B. Constant-Q Transform (CQT)
        cqt_spec = np.abs(librosa.cqt(y=y, sr=sr))
        log_cqt_spec = librosa.amplitude_to_db(cqt_spec, ref=np.max)
        cqt_features = np.mean(log_cqt_spec.T, axis=0)

        # C. Chroma Features
        chroma_spec = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_features = np.mean(chroma_spec.T, axis=0)

        # combined_features = np.hstack((mel_features, cqt_features, chroma_features))
        combined_features = np.hstack((mel_features,cqt_features))


        # 4. Append to lists ONLY upon success
        X.append(combined_features)
        u.append(label)
        folds_list.append(fold_num) # <-- NEW: Append the fold

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 5. Convert lists to numpy arrays
X = np.array(X)
u = np.array(u)
folds_array = np.array(folds_list) # <-- NEW: Convert folds to array

print("\n--- Extraction Complete ---")
print(f"Shape of X: {X.shape}")
print(f"Shape of u: {u.shape}")
print(f"Shape of folds: {folds_array.shape}")

Starting feature extraction for 501 files...


/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed 50 / 501 files...
Processed 100 / 501 files...
Processed 150 / 501 files...
Processed 200 / 501 files...
Processed 250 / 501 files...
Processed 300 / 501 files...
Processed 350 / 501 files...
Processed 400 / 501 files...
Processed 450 / 501 files...
Processed 500 / 501 files...

--- Extraction Complete ---
Shape of X: (501, 212)
Shape of u: (501,)
Shape of folds: (501,)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Encode text labels to integers
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)

# Array to store the accuracy of each fold
fold_accuracies = []

print("\n--- Starting 5-Fold Cross Validation ---")

# 2. Loop through the 5 folds
for test_fold in range(1, 6):
    print(f"\n--- Processing Fold {test_fold} as Test Set ---")

    # Create boolean masks to separate train and test data
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    # Split the data into Train (4 folds) and Test (1 fold)
    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples.")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and Train the model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Test the model
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    fold_accuracies.append(acc)

    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

# Final Results
print("\n--- Final Results ---")
print(f"Accuracies across the 5 folds: {[round(a, 4) for a in fold_accuracies]}")
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f}")

In [ ]:
classical model

In [1]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Encode text labels (like 'cop', 'hea') to integers (0, 1, 2...)
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)
num_classes = len(encoder.classes_)
print(f"Classes mapping: {dict(enumerate(encoder.classes_))}")

# Array to store the accuracy of each fold
rf_fold_accuracies = []

print("\n=== Starting 5-Fold Cross Validation (Random Forest) ===")

# 2. Loop through the 5 folds
for test_fold in range(1, 6):
    print(f"\n--- Fold {test_fold} ---")

    # Create boolean masks to separate train and test data
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    # Split the data into Train (4 folds) and Test (1 fold)
    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    # 3. Scale the features (Important so large feature values don't dominate)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Initialize and Train the model
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train_scaled, y_train)

    # 5. Evaluate the model
    predictions = rf_model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    rf_fold_accuracies.append(acc)

    print(f"Accuracy for Fold {test_fold}: {acc:.4f}")

# Final Results
print("\n=== Final Random Forest Results ===")
print(f"Accuracies across 5 folds: {[round(a, 4) for a in rf_fold_accuracies]}")
print(f"Average Cross-Validation Accuracy: {np.mean(rf_fold_accuracies):.4f}")

NameError: name 'u' is not defined

DL 1D CNN

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Encode labels if not already done
encoder = LabelEncoder()
u_encoded = encoder.fit_transform(u)
num_classes = len(encoder.classes_)

# Function to build a fresh, untrained model for each fold
def create_1d_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=128, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5), # Prevents overfitting

        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_fold_accuracies = []

print("\n=== Starting 5-Fold Cross Validation (1D CNN) ===")

for test_fold in range(1, 6):
    print(f"\n--- Fold {test_fold} ---")

    # Split the data using masks
    test_mask = (folds_array == test_fold)
    train_mask = (folds_array != test_fold)

    X_train, y_train = X[train_mask], u_encoded[train_mask]
    X_test, y_test = X[test_mask], u_encoded[test_mask]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Reshape data for Conv1D (Samples, Features, 1)
    X_train_cnn = np.expand_dims(X_train_scaled, axis=2)
    X_test_cnn = np.expand_dims(X_test_scaled, axis=2)

    # Build a fresh model for this fold
    input_shape = (X_train_cnn.shape[1], 1)
    cnn_model = create_1d_cnn(input_shape, num_classes)

    # Train the model (verbose=0 hides the epoch-by-epoch progress bar to keep output clean)
    print("Training...")
    cnn_model.fit(X_train_cnn, y_train, epochs=30, batch_size=32, verbose=0)

    # Evaluate on the test fold
    test_loss, test_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
    cnn_fold_accuracies.append(test_acc)

    print(f"Accuracy for Fold {test_fold}: {test_acc:.4f}")

# Final Results
print("\n=== Final 1D CNN Results ===")
print(f"Accuracies across 5 folds: {[round(a, 4) for a in cnn_fold_accuracies]}")
print(f"Average Cross-Validation Accuracy: {np.mean(cnn_fold_accuracies):.4f}")